# TC_01: Random forest clean baseline
Full diagnostic pipeline on clean data using RF model.
UQ: Deep Ensemble,
XAI: SHAP + LIME

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_tabular_data
from src.data_diagnostics.quality_checks import check_tabular_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.tabular_models import train_random_forest,evaluate_model_rf_xgb
from src.inference_diagnostics.uncertainty import deep_ensemble_prediction_sklern
from src.inference_diagnostics.explainability import shap_tab,lime_tab,calculate_consistency_tabular
from src.inference_diagnostics.flagging import assign_flags,evaluate_flags
from src.utils.visualization import plot_accuracy_comparison, plot_uncertainty_distribution, plot_flag_distribution
from src.utils.report_generator import generate_report,save_report
from src.utils.config_loader import load_config, get_tabular_config, save_threshold
from src.utils.per_sample_logger import save_per_sample, build_experiment_id

config = load_config()
tracker = PerformanceTracker()

In [2]:
# Identify this run
dataset_short = get_tabular_config(config)['short_name']
model_name = 'rf'
test_case = 'TC01'
experiment_id = build_experiment_id(dataset_short, model_name, test_case)
print(f"Experiment: {experiment_id}")

Experiment: adult_rf_TC01


### Load data and EDA

In [3]:
X_train, X_test, y_train, y_test = load_tabular_data(config)
report = check_tabular_quality(X_train, y_train, config)
plot_class_distribution(report, f"RF {get_tabular_config(config)['name']} Baseline", config)
plot_correlation(report, f"RF {get_tabular_config(config)['name']} Baseline", config)
plot_feature_boxplots(X_train, f"RF {get_tabular_config(config)['name']} Baseline", config)

Loading dataset.
Dataset: UCI Adult Income
Duplicate rows kept (6374 found).
Found 6465 missing values. Filling with mode.
Data loaded for 39073 train, 9769 test
 Features: 12
EDA started for tabular data.
Samples: 39073, Features: 12
Class distribution: {0: 29724, 1: 9349}
Imbalance ratio: 0.3145
Duplicate rows: 5422
Total outlier count: 5915
EDA completed for UCI Adult Income
Saved: results/figures\class_distribution_rf_uci_adult_income_baseline.png
Saved: results/figures\correlation_rf_uci_adult_income_baseline.png
Saved: results/figures\boxplot_rf_uci_adult_income_baseline.png


### Train random forest baseline

In [4]:
# Random Forest
tracker.start_performance_track()
rf_model = train_random_forest(X_train, y_train, config)
tracker.stop_performance_track("Random forest training")

tracker.start_performance_track()
rf_accuracy, rf_prediction, rf_report = evaluate_model_rf_xgb(rf_model, X_test, y_test)
tracker.stop_performance_track("Random forest evaluation")

Random Forest training started.
Random Forest training completed.
Random forest training: Time:0.23s, Memory:34.96MB
Model evaluation started for RF,XGB
{'0': {'precision': 0.8855746334127084, 'recall': 0.9508814426053022, 'f1-score': 0.9170668397144711, 'support': 7431.0}, '1': {'precision': 0.7960893854748603, 'recall': 0.6094952951240377, 'f1-score': 0.690406976744186, 'support': 2338.0}, 'accuracy': 0.8691780120790255, 'macro avg': {'precision': 0.8408320094437843, 'recall': 0.7801883688646699, 'f1-score': 0.8037369082293286, 'support': 9769.0}, 'weighted avg': {'precision': 0.8641582643187696, 'recall': 0.8691780120790255, 'f1-score': 0.8628206774026146, 'support': 9769.0}}
Random forest evaluation: Time:0.04s, Memory:6.77MB


{'operation': 'Random forest evaluation',
 'time_seconds': 0.04,
 'memory_usage': 6.77}

### Uncertainty Quantification (Deep Ensembles)

In [5]:
tracker.start_performance_track()
de_means_probabilities, de_uncertainty = deep_ensemble_prediction_sklern(train_random_forest, X_train, y_train, X_test, config)
tracker.stop_performance_track("RF Deep Ensemble prediction")

de_threshold = round(float(np.percentile(de_uncertainty, 90)), 4)
save_threshold(config, dataset_short, model_name, 'de', de_threshold)
de_y_prediction = de_means_probabilities.argmax(axis=1)
print(f"Deep Ensembl uncertainty status:")
print(f"Mean: {de_uncertainty.mean()}")
print(f"Max: {de_uncertainty.max()}")
print(f" 90th percentile (threshold): {de_threshold}")

plot_uncertainty_distribution(de_uncertainty, "RF Deep Ensemble", de_threshold, config)

Deep Ensemble started for tabular and sklern.
Training model with seed 0
Random Forest training started.
Random Forest training completed.
Training model with seed 1
Random Forest training started.
Random Forest training completed.
Training model with seed 2
Random Forest training started.
Random Forest training completed.
Training model with seed 3
Random Forest training started.
Random Forest training completed.
Training model with seed 4
Random Forest training started.
Random Forest training completed.
Deep Ensemble finished for tabular sklern.
RF Deep Ensemble prediction: Time:1.19s, Memory:2.54MB
Threshold saved: adult_rf_de = 0.0
Deep Ensembl uncertainty status:
Mean: 3.910871014874296e-17
Max: 1.1991167267330158e-16
 90th percentile (threshold): 0.0
Saved: results/figures\uncertainty_distribution_rf_deep_ensemble.png


### Explainability (SHAP and LIME)

In [6]:
# SHAP
tracker.start_performance_track()
shap_values, shap_samples = shap_tab(rf_model, X_train, X_test, config, is_pytorch = False)
tracker.stop_performance_track("RF SHAP")
print(f"SHAP values shape: {shap_values.shape}")

SHAP started.
SHAP finished. Explained: 200 samples.
RF SHAP: Time:6.04s, Memory:10.20MB
SHAP values shape: (200, 12, 2)


In [7]:
# LIME
tracker.start_performance_track()
lime_explanations, lime_samples = lime_tab(rf_model, X_train, X_test, config, is_pytorch = False)
tracker.stop_performance_track("RF LIME")
print(f"LIME explanations: {len(lime_explanations)}")

LIME started.


C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\pra

Explained 50 / 200 samples.


C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\pra

Explained 100 / 200 samples.


C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\pra

Explained 150 / 200 samples.


C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\pra

Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
RF LIME: Time:9.20s, Memory:0.00MB
LIME explanations: 200


In [8]:
# Calculate consistency
feature_names = list(X_train.columns)
consistency_scores = calculate_consistency_tabular(shap_values, lime_explanations, feature_names, top=5)
print(f"Mean consistency: {consistency_scores.mean()}")
print(f"Min consistency: {consistency_scores.min()}")
print(f"Max consistency: {consistency_scores.max()}")


Calculate consistency tabular.
Mean consistency: 0.606
Min consistency: 0.2
Max consistency: 1.0


### Flagging

In [9]:
# Real uncertainty value for RF is near 0 and meaning less ofr flagging system.
rf_uq_threshold = float(de_uncertainty.max()) + 1.0
rf_flags = assign_flags(de_uncertainty[:len(consistency_scores)], consistency_scores, rf_uq_threshold, 0.5)

print("RF flagging results:")
rf_flagging_result = evaluate_flags(rf_flags, rf_prediction[:len(consistency_scores)], y_test[:len(consistency_scores)])

plot_flag_distribution(rf_flags, 'RF Baseline', config)

RF flagging results:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 33 Accuracy:81.8%
GREEN: Count: 167 Accuracy:90.4%
Flagging system is working
Saved: results/figures\flagging_distribution_rf_baseline.png


In [10]:
# Per sample results for the ablation and metrics analysis
save_per_sample(
    config,
    experiment_id,
    y_true=y_test,
    y_pred=rf_prediction,
    mc_uncertainty=None,
    de_uncertainty=de_uncertainty,
    consistency=consistency_scores,
)

Per sample results saved: results/per_sample\adult_rf_TC01.csv (200 rows)


,sample_id,y_true,y_pred,correct,mc_uncertainty,de_uncertainty,consistency
0,0,0,0,1,NaN,6.587563e-17,0.6
1,1,0,0,1,NaN,2.862593e-17,0.8
2,2,0,0,1,NaN,7.026516e-17,0.6
3,3,0,0,1,NaN,6.792382e-17,0.6
4,4,1,1,1,NaN,5.993368e-17,0.6
...,...,...,...,...,...,...,...
195,195,0,0,1,NaN,4.965068e-17,0.6
196,196,0,1,0,NaN,5.993368e-17,0.6
197,197,0,0,1,NaN,0.000000e+00,0.8
198,198,1,1,1,NaN,0.000000e+00,0.4


### Save baseline report


In [11]:
rf_baseline = {
    'model': 'Random Forest',
    'accuracy': round(rf_accuracy, 4),
    'classification_report': rf_report,
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold,
        'findings': 'Near zero uncertainty and RF internal ensemble makes external DE redundant'
    },
    'consistency':{
        'mean': round(float(consistency_scores.mean()), 4),
        'min': round(float(consistency_scores.min()), 4),
        'max': round(float(consistency_scores.max()), 4)
    },
    'flagging': rf_flagging_result,
    'performance': tracker.get_results()
}

report_output = generate_report(config, f"{get_tabular_config(config)['name']} - RF clean baseline", stage1_result=rf_baseline)
save_report(report_output, f'{dataset_short.upper()}_TC_01_Random_Forest_Clean_Baseline.json', config)

Generating report.
Diagnostic report generated.
Saving report.
